In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, IntegerType, LongType

### Silver to Gold Layer Transformation: BI-Ready Fact Table
##### **Catalog:** `ecommerce` | **Schema:** `gold` | **Table:** `gld_fact_order_items`

##### Extract: Read Silver Layer Data

In [0]:
slv_df = spark.table(f"ecommerce.silver.slv_order_items")

##### Reference Data: Foreign Exchange (FX) Rates Table Creation

In [0]:
fx_rates_data = [
    ("INR", 1.00),
    ("AED", 24.18),
    ("AUD", 57.55),
    ("CAD", 62.93),
    ("GBP", 117.98),
    ("SGD", 68.18),
    ("USD", 88.29),
]

fx_schema = ["currency_code", "inr_rate"]
fx_rates_df = spark.createDataFrame(fx_rates_data, schema=fx_schema)

##### Transform: Business Logic & Feature Engineering

In [0]:
gold_transformed_df = (
    slv_df
    # Clean string inputs for join consistency
    .withColumn(
        "clean_currency",
        F.upper(F.trim(F.col("unit_price_currency")))
    )
    # Cast tax_amount to numeric type safely
    .withColumn(
        "tax_amount_num",
        F.col("tax_amount").cast(DoubleType())
    )
    # Measure Calculations
    .withColumn(
        "gross_amount",
        F.col("quantity") * F.col("unit_price")
    )
    .withColumn(
        "discount_amount",
        F.ceil(F.col("gross_amount") * (F.col("discount_pct") / F.lit(100.0))).cast(LongType())
    )
    .withColumn(
        "sale_amount",
        F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount_num")
    )
    # Surrogate Keys & Flags
    .withColumn(
        "date_id",
        F.date_format(F.to_date(F.col("dt")), "yyyyMMdd").cast(IntegerType())
    )
    .withColumn(
        "coupon_flag",
        F.when(F.col("coupon_code").isNotNull() & (F.trim(F.col("coupon_code")) != ""), F.lit(1)).otherwise(F.lit(0))
    )
    # Currency Conversion Join
    .join(
        fx_rates_df,
        F.col("clean_currency") == fx_rates_df.currency_code,
        "left"
    )
    .withColumn(
        "sale_amount_inr",
        F.ceil(F.col("sale_amount") * F.coalesce(F.col("inr_rate"), F.lit(1.0))).cast(LongType())
    )
)

##### Schema Alignment & Final Projection (Fact Model)

In [0]:
orders_gold_fact_df = gold_transformed_df.select(
    F.col("date_id"),
    F.col("dt").cast("date").alias("transaction_date"),
    F.col("order_ts").cast("timestamp").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").cast(IntegerType()).alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity").cast(IntegerType()),
    F.col("unit_price").cast(DoubleType()),
    F.col("gross_amount").cast(DoubleType()),
    F.col("discount_pct").alias("discount_percent").cast(DoubleType()),
    F.col("discount_amount"),
    F.col("tax_amount_num").alias("tax_amount"),
    F.col("sale_amount").alias("net_amount").cast(DoubleType()),
    F.col("sale_amount_inr").alias("net_amount_inr")
)

##### Load: Write Delta Table to Gold Storage

In [0]:
(
    orders_gold_fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"ecommerce.gold.gld_fact_order_items")
)

##### Data Quality & Sanity Check

In [0]:
# record_count = spark.table(f"ecommerce.gold.gld_fact_order_items").count()
# print(f"Total records successfully written to Gold Layer: {record_count}")
# display(spark.table(f"ecommerce.gold.gld_fact_order_items").limit(5))